# Bike Rental Data Preprocessing Workflow Design


## 1 Introduction
----

This notebook explores the raw datasets used for the Week 2 bike rental data pipeline assignment.

The goal of this analysis is not to train a machine learning model yet. Instead, the focus is to understand the available data sources, inspect their quality, and design the preprocessing steps needed to create a final ML-ready dataset.

The project data consists of four CSV files:

- registered bike rentals
- direct pickup bike rentals
- weather data
- holiday calendar

The rental datasets contain individual rental events, while the weather and holiday datasets provide additional contextual information that may help explain bike demand patterns.

During this Notebook, we will:

- inspect schemas, data types, missing values, and duplicates
- validate datetime columns and time coverage
- analyze rental activity and location structure
- prototype hourly rental aggregation
- validate how weather and holiday data can be joined
- define the expected structure of the final prepared dataset

The main outcome of this notebook is a clear preprocessing plan that can later be implemented as a reproducible Dagster pipeline.


## 2. Imports & Setup

---

The analysis in this notebook uses pandas for data manipulation and exploration, 
matplotlib and seaborn for visualizations, and pathlib for handling file paths.


In [1]:
from pathlib import Path

import pandas as pd
import seaborn as sns

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

sns.set_theme(style="whitegrid")

In [3]:
DATA_DIR = Path("../../data/raw")

REGISTERED_RENTALS_PATH = DATA_DIR / "registered_bike_rentals.csv"
DIRECT_PICKUPS_PATH = DATA_DIR / "direct_pickup_bike_rentals.csv"
WEATHER_PATH = DATA_DIR / "weather.csv"
HOLIDAYS_PATH = DATA_DIR / "holidays.csv"

## 3. Dataset Loading

---

In [4]:
registered_rentals_df = pd.read_csv(REGISTERED_RENTALS_PATH)
direct_pickups_df = pd.read_csv(DIRECT_PICKUPS_PATH)
weather_df = pd.read_csv(WEATHER_PATH)
holidays_df = pd.read_csv(HOLIDAYS_PATH)

## 4. Initial Dataset Inspection

---

The goal of this section is to inspect the structure and quality of each dataset
before designing the preprocessing pipeline. This includes:
- checking dataset
- dimensions
- column types
- missing values
- duplicate rows
- example records.

In [5]:
def inspect_dataframe(df: pd.DataFrame, name: str) -> None:
    """Print basic inspection information for a DataFrame."""
    print(f"\n{'=' * 60}")
    print(f"{name}")
    print(f"{'=' * 60}")

    print("\nShape:")
    print(df.shape)

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nDuplicate rows:")
    print(df.duplicated().sum())

    display(df.head())

## 4.1 Registered Rentals Inspection

In [6]:
inspect_dataframe(registered_rentals_df, "Registered Rentals")


Registered Rentals

Shape:
(2672662, 4)

Data types:
id             int64
datetime         str
user_id        int64
location_id    int64
dtype: object

Missing values:
id             0
datetime       0
user_id        0
location_id    0
dtype: int64

Duplicate rows:
0


,id,datetime,user_id,location_id
0,1,2011-01-01 00:05:09,158,16
1,2,2011-01-01 00:05:21,262,18
2,3,2011-01-01 00:05:39,68,18
3,4,2011-01-01 00:12:05,12,9
4,5,2011-01-01 00:25:58,91,11


### Observations

- The dataset contains approximately 2.67 million registered rental events, making it the largest operational dataset in the project.
- The schema is simple and fully structured, consisting of identifiers, timestamps, and location information.
- No missing values or duplicate rows were detected.
- The `datetime` column is currently stored as a string and will need to be converted to a proper datetime type for time-based processing and aggregation.
- Each row appears to represent a single bike rental event.

## 3.2 Direct Pickup Rentals Inspection

In [7]:
inspect_dataframe(direct_pickups_df, "Direct Pickups")


Direct Pickups

Shape:
(620017, 4)

Data types:
id             int64
datetime         str
user_id        int64
location_id    int64
dtype: object

Missing values:
id             0
datetime       0
user_id        0
location_id    0
dtype: int64

Duplicate rows:
0


,id,datetime,user_id,location_id
0,1,2011-01-01 00:24:04,232,2
1,2,2011-01-01 00:30:19,54,14
2,3,2011-01-01 00:39:08,201,5
3,4,2011-01-01 01:01:12,298,13
4,5,2011-01-01 01:02:37,23,14


### Observations

- The dataset contains approximately 620 thousand direct pickup rental events.
- The schema matches the registered rentals dataset, which simplifies later combination of both operational sources.
- No missing values or duplicate rows were detected.
- The `datetime` column is currently stored as a string and requires datetime conversion before preprocessing.
- The dataset appears to contain one row per direct rental.

## 3.3 Weather Data Inspection

In [8]:
inspect_dataframe(weather_df, "Weather Data")


Weather Data

Shape:
(17379, 7)

Data types:
id                           int64
datetime                       str
conditions                     str
temperature_c              float64
perceived_temperature_c    float64
humidity                   float64
windspeed_kmh              float64
dtype: object

Missing values:
id                         0
datetime                   0
conditions                 0
temperature_c              0
perceived_temperature_c    0
humidity                   0
windspeed_kmh              0
dtype: int64

Duplicate rows:
0


,id,datetime,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh
0,1,2011-01-01 00:00:00,clear,3.3,3.0,81.0,0.0
1,2,2011-01-01 01:00:00,clear,2.3,2.0,80.0,0.0
2,3,2011-01-01 02:00:00,clear,2.3,2.0,80.0,0.0
3,4,2011-01-01 03:00:00,clear,3.3,3.0,75.0,0.0
4,5,2011-01-01 04:00:00,clear,3.3,3.0,75.0,0.0


### Observations

- The weather dataset contains approximately 17 thousand rows and includes hourly weather-related measurements.
- No missing values or duplicate rows were detected.
- The dataset includes both categorical and numerical weather features, such as weather conditions, temperature, humidity, and wind speed.
- The `datetime` column is currently stored as a string and will need to be converted to a datetime type.
- The `conditions` column is currently stored as a string categorical feature and will likely require encoding or categorization before later machine learning usage.
- Numerical weather measurements are already stored using appropriate floating-point data types.
- The dataset appears well-suited for later enrichment of the rental activity data.

## 3.4 Holidays Inspection

In [9]:
inspect_dataframe(holidays_df, "Holiday Data")


Holiday Data

Shape:
(21, 3)

Data types:
id         int64
date         str
holiday      str
dtype: object

Missing values:
id         0
date       0
holiday    0
dtype: int64

Duplicate rows:
0


,id,date,holiday
0,1,2011-01-17,"Dr. Martin Luther King, Jr.'s Birthday"
1,2,2011-02-21,Washington's Birthday
2,3,2011-04-15,D.C. Emancipation Day (observed)
3,4,2011-05-30,Memorial Day
4,5,2011-07-04,Independence Day


### Observations

- The holiday dataset is small and contains 21 holiday entries.
- No missing values or duplicate rows were detected.
- The dataset consists of a date column and a descriptive holiday name.
- The `date` column is currently stored as a string and requires conversion to a date or datetime type before joining.
- The dataset will likely be used to derive a binary holiday indicator feature for the final ML-ready dataset.

## 5. Datetime Processing & Validation

---


Before the datasets can be aggregated and combined, all date/time columns must be
converted from strings to proper datetime objects. This enables time-based
operations such as 
- hourly aggregation
- filtering
- dataset joins.

In this section, the datetime columns are converted and validated to ensure that the timestamps align correctly across all datasets.

In [10]:
registered_rentals_df["datetime"] = pd.to_datetime(
    registered_rentals_df["datetime"], errors="coerce"
)
direct_pickups_df["datetime"] = pd.to_datetime(direct_pickups_df["datetime"], errors="coerce")
weather_df["datetime"] = pd.to_datetime(weather_df["datetime"], errors="coerce")
holidays_df["date"] = pd.to_datetime(holidays_df["date"], errors="coerce")

In [11]:
registered_rentals_df["datetime"].head()
registered_rentals_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2672662 entries, 0 to 2672661
Data columns (total 4 columns):
 #   Column       Dtype         
---  ------       -----         
 0   id           int64         
 1   datetime     datetime64[us]
 2   user_id      int64         
 3   location_id  int64         
dtypes: datetime64[us](1), int64(3)
memory usage: 81.6 MB


In [12]:
print(
    "Registered rentals invalid timestamps:",
    registered_rentals_df["datetime"].isna().sum(),
)

print(
    "Direct pickups invalid timestamps:",
    direct_pickups_df["datetime"].isna().sum(),
)

print(
    "Weather invalid timestamps:",
    weather_df["datetime"].isna().sum(),
)

print(
    "Holiday invalid dates:",
    holidays_df["date"].isna().sum(),
)

Registered rentals invalid timestamps: 0
Direct pickups invalid timestamps: 0
Weather invalid timestamps: 0
Holiday invalid dates: 0


In [13]:
print(
    "Registered rentals date range:",
    registered_rentals_df["datetime"].min(),
    "->",
    registered_rentals_df["datetime"].max(),
)

print(
    "Direct pickups date range:",
    direct_pickups_df["datetime"].min(),
    "->",
    direct_pickups_df["datetime"].max(),
)

print(
    "Weather date range:",
    weather_df["datetime"].min(),
    "->",
    weather_df["datetime"].max(),
)

print(
    "Holiday date range:",
    holidays_df["date"].min(),
    "->",
    holidays_df["date"].max(),
)

Registered rentals date range: 2011-01-01 00:05:09 -> 2012-12-31 23:55:53
Direct pickups date range: 2011-01-01 00:24:04 -> 2012-12-31 23:51:18
Weather date range: 2011-01-01 00:00:00 -> 2012-12-31 23:00:00
Holiday date range: 2011-01-17 00:00:00 -> 2012-12-25 00:00:00


### Observations

- All datetime and date columns were successfully converted without invalid values.
- The rentals, direct pickups, and weather datasets cover a consistent time period from January 2011 to December 2012.
- The holiday dataset covers the same general project timeframe and can later be used to derive holiday-related features.

# 6. Aggregation & Temporal Structure Validation

---

This section validates the temporal structure of the operational and weather
datasets before implementing the preprocessing pipeline.

The focus is on verifying that the rental data can be aggregated into hourly
time windows and that the weather data aligns sufficiently with the expected
hourly granularity required for later dataset joins and feature engineering.

In [14]:
registered_rentals_df["datetime_hour"] = registered_rentals_df["datetime"].dt.floor("h")

direct_pickups_df["datetime_hour"] = direct_pickups_df["datetime"].dt.floor("h")

print(
    "Registered rental hours:",
    registered_rentals_df["datetime_hour"].nunique(),
)

print(
    "Direct pickup hours:",
    direct_pickups_df["datetime_hour"].nunique(),
)

Registered rental hours: 17355
Direct pickup hours: 15798


### Observation

- Rental events span a large number of unique hourly time windows across the two-year project period. 17355 of approx. 17520.
- The registered rentals dataset covers nearly the full hourly timeline, while direct pickups appear slightly sparser.
- The results confirm that aggregating rental activity into hourly intervals is a suitable preprocessing strategy for the final dataset.

In [15]:
weather_time_diff = weather_df["datetime"].sort_values().diff().value_counts()

weather_time_diff

datetime
0 days 01:00:00    17303
0 days 02:00:00       64
0 days 03:00:00        6
0 days 13:00:00        1
0 days 23:00:00        1
0 days 07:00:00        1
0 days 14:00:00        1
1 days 13:00:00        1
Name: count, dtype: int64

In [16]:
expected_hours = pd.date_range(
    start=weather_df["datetime"].min(),
    end=weather_df["datetime"].max(),
    freq="h",
)

missing_weather_hours = expected_hours.difference(weather_df["datetime"])

print(
    "Missing weather hours:",
    len(missing_weather_hours),
)

Missing weather hours: 165


### Observation


- The weather dataset is primarily structured at an hourly granularity.
- Most consecutive weather records are spaced one hour apart, but some larger temporal gaps exist.
- Validation against a complete hourly timestamp range identified 165 missing hourly weather observations.
- Missing weather records may later lead to incomplete joins with aggregated rental activity and will require consideration during preprocessing.

# 7. Prototype Aggregation

---

In this section, the planned preprocessing workflow is prototyped by transforming
individual rental events into aggregated hourly rental activity.

The goal is to validate the core aggregation strategy that will later be used
inside the Dagster pipeline. Rental events are grouped into hourly time windows
and aggregated by location to create the foundation of the final ML-ready
dataset.

In [17]:
# combnine rental and direct pickup dfs

registered_rentals_df["is_registered"] = True
direct_pickups_df["is_registered"] = False

rentals_df = pd.concat(
    [registered_rentals_df, direct_pickups_df],
    ignore_index=True,
)

In [18]:
# group by datetime_hour, location_id, to get counts of rentals per hour and location
# use unstack for is_registered to get separate columns for direct pickups and registered rentals

hourly_rentals_df = (
    rentals_df.groupby(["datetime_hour", "location_id", "is_registered"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

hourly_rentals_df.columns = [
    "datetime_hour",
    "location_id",
    "direct_pickups",
    "registered_rentals",
]

hourly_rentals_df["total_rentals"] = (
    hourly_rentals_df["direct_pickups"] + hourly_rentals_df["registered_rentals"]
)

hourly_rentals_df.head(5)

,datetime_hour,location_id,direct_pickups,registered_rentals,total_rentals
0,2011-01-01,2,1,0,1
1,2011-01-01,3,0,1,1
2,2011-01-01,4,0,2,2
3,2011-01-01,5,1,0,1
4,2011-01-01,9,0,1,1


### Observations

- The aggregation successfully transformed individual rental events into hourly rental counts grouped by location.
- Separate aggregation of registered rentals and direct pickups preserves rental type count (direct_pickups, registed_pickups) kept for operational analysis and dataset completeness will not be used as predictive input
features (target leakage).
- The resulting dataset structure aligns with the planned final ML-ready dataset, where each row represents rental activity for a specific location during a specific hourly time window.

# 8. Feature Engineering Planning

---

This section outlines additional features that may improve the usefulness of the final ML-ready dataset.

In [19]:
hourly_rentals_df["hour"] = hourly_rentals_df["datetime_hour"].dt.hour

hourly_rentals_df["weekday"] = hourly_rentals_df["datetime_hour"].dt.weekday

hourly_rentals_df["month"] = hourly_rentals_df["datetime_hour"].dt.month

hourly_rentals_df["is_weekend"] = hourly_rentals_df["weekday"] >= 5

In [20]:
hourly_rentals_df.head()

,datetime_hour,location_id,direct_pickups,registered_rentals,total_rentals,hour,weekday,month,is_weekend
0,2011-01-01,2,1,0,1,0,5,1,True
1,2011-01-01,3,0,1,1,0,5,1,True
2,2011-01-01,4,0,2,2,0,5,1,True
3,2011-01-01,5,1,0,1,0,5,1,True
4,2011-01-01,9,0,1,1,0,5,1,True


### Observation

Features Successfully added:

- hour
- weekday
- month
- is_weekend

# 9. Weather & Holiday Join Validation

---

This section validates that the aggregated rental activity can be successfully enriched with weather and holiday information.

In [21]:
hourly_rentals_df = hourly_rentals_df.merge(
    weather_df,
    left_on="datetime_hour",
    right_on="datetime",
    how="left",
)

hourly_rentals_df = hourly_rentals_df.drop(columns=["id", "datetime"])

In [22]:
hourly_rentals_df.head()

,datetime_hour,location_id,direct_pickups,registered_rentals,total_rentals,hour,weekday,month,is_weekend,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh
0,2011-01-01,2,1,0,1,0,5,1,True,clear,3.3,3.0,81.0,0.0
1,2011-01-01,3,0,1,1,0,5,1,True,clear,3.3,3.0,81.0,0.0
2,2011-01-01,4,0,2,2,0,5,1,True,clear,3.3,3.0,81.0,0.0
3,2011-01-01,5,1,0,1,0,5,1,True,clear,3.3,3.0,81.0,0.0
4,2011-01-01,9,0,1,1,0,5,1,True,clear,3.3,3.0,81.0,0.0


In [23]:
hourly_rentals_df = hourly_rentals_df.merge(
    holidays_df[["date", "holiday"]],
    left_on="datetime_hour",
    right_on="date",
    how="left",
)

In [24]:
hourly_rentals_df["is_holiday"] = hourly_rentals_df["holiday"].notna()

hourly_rentals_df = hourly_rentals_df.drop(columns=["date", "holiday"])

In [25]:
hourly_rentals_df.head()

,datetime_hour,location_id,direct_pickups,registered_rentals,total_rentals,hour,weekday,month,is_weekend,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh,is_holiday
0,2011-01-01,2,1,0,1,0,5,1,True,clear,3.3,3.0,81.0,0.0,False
1,2011-01-01,3,0,1,1,0,5,1,True,clear,3.3,3.0,81.0,0.0,False
2,2011-01-01,4,0,2,2,0,5,1,True,clear,3.3,3.0,81.0,0.0,False
3,2011-01-01,5,1,0,1,0,5,1,True,clear,3.3,3.0,81.0,0.0,False
4,2011-01-01,9,0,1,1,0,5,1,True,clear,3.3,3.0,81.0,0.0,False


In [26]:
hourly_rentals_df["temperature_c"].isna().sum()

np.int64(0)

### Observations

- All aggregated rental rows were successfully enriched with weather and holiday data.
- Although the weather dataset contains missing hourly observations, these hours do not overlap with existing rental activity.

## 10. Final Dataset Design

---

Based on the previous exploration and preprocessing validation steps, the final dataset structure can now be defined.

The goal of the final dataset is to represent hourly bike rental activity per location together with temporal and contextual features that may help later machine learning models learn rental demand patterns.


## 10.1 Planned Dataset Granularity

Each row in the final dataset represents:

- one location
- during one hourly time window

The primary dataset identifiers are therefore:

- datetime_hour
- location_id


## 10.2 Planned Target Variable

The primary target variable for later machine learning workflows is:

- total_rentals


## 10.3 Planned Feature Columns

### Temporal Features
- hour
- weekday
- month
- is_weekend
- is_holiday

### Location Features
- location_id

### Weather Features
- conditions
- temperature_c
- perceived_temperature_c
- humidity
- windspeed_kmh

### Operational & Analysis Features
- registered_rentals *(Note: Analytical only; excluded from model training)*
- direct_pickups *(Note: Analytical only; excluded from model training)*

---

> ⚠️ **Target Leakage Warning:** The operational features `registered_rentals` and `direct_pickups` sum directly to the target variable `total_rentals`. They are retained in the final dataset exclusively for historical analysis, debugging, and operational completeness. They **must not** be used as input features during model training, as this would cause target leakage and trivialize the forecasting task.

---

### Final Prototype Dataset Inspection


In [27]:
hourly_rentals_df.head()

,datetime_hour,location_id,direct_pickups,registered_rentals,total_rentals,hour,weekday,month,is_weekend,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh,is_holiday
0,2011-01-01,2,1,0,1,0,5,1,True,clear,3.3,3.0,81.0,0.0,False
1,2011-01-01,3,0,1,1,0,5,1,True,clear,3.3,3.0,81.0,0.0,False
2,2011-01-01,4,0,2,2,0,5,1,True,clear,3.3,3.0,81.0,0.0,False
3,2011-01-01,5,1,0,1,0,5,1,True,clear,3.3,3.0,81.0,0.0,False
4,2011-01-01,9,0,1,1,0,5,1,True,clear,3.3,3.0,81.0,0.0,False


In [28]:
hourly_rentals_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 312386 entries, 0 to 312385
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   datetime_hour            312386 non-null  datetime64[us]
 1   location_id              312386 non-null  int64         
 2   direct_pickups           312386 non-null  int64         
 3   registered_rentals       312386 non-null  int64         
 4   total_rentals            312386 non-null  int64         
 5   hour                     312386 non-null  int32         
 6   weekday                  312386 non-null  int32         
 7   month                    312386 non-null  int32         
 8   is_weekend               312386 non-null  bool          
 9   conditions               312386 non-null  str           
 10  temperature_c            312386 non-null  float64       
 11  perceived_temperature_c  312386 non-null  float64       
 12  humidity                 31

## 11. Summary & Next Steps

---

In this assignment, the available operational, weather, and holiday datasets
were explored, validated, and transformed into a unified preprocessing
prototype for later machine learning workflows.

The preprocessing steps included:
- loading and inspecting the source datasets
- validating temporal consistency and data quality
- aggregating rental events into hourly activity per location
- deriving temporal features
- enriching the dataset with weather and holiday information

The resulting prototype dataset now provides a structured hourly feature table
that can later be used for forecasting and machine learning tasks.


### Next Steps

The next stage of the project will focus on translating the explored
preprocessing logic into a reusable Dagster pipeline.

This will include:
- implementing the preprocessing steps as Dagster assets
- organizing the project into reusable modules
- materializing the final dataset automatically
- preparing the pipeline for later model training workflows